# The number you would have reported

MichAl Academy, lesson 2.11.

Run each cell with **Shift+Enter**.

Every accuracy figure in this track was produced by holding back some rows and
scoring the rest. This notebook asks what happens if you hold back a different
set of rows, and the answer is larger than most of the differences we have been
comparing models on.

Then it asks the harder question. If you try seven settings and keep the one
that cross-validated best, is that best score still an honest estimate? It is
not, and the amount it lies by is measurable.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV,
                                     learning_curve)

SEED = 0
X, y = load_breast_cancer(return_X_y=True)
print(f"{len(y)} rows, {X.shape[1]} features, {y.mean():.1%} positive")


def model(C=1.0):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))


## 1. Change nothing but the seed

Same data. Same model. Same 80/20 proportion. Two hundred times, changing only
which rows land in the held-back fifth.


In [ ]:
scores = []
for seed in range(200):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    scores.append(model().fit(X_tr, y_tr).score(X_te, y_te))
scores = np.array(scores)

print(f"mean    {scores.mean():.4f}")
print(f"sd      {scores.std():.4f}")
print(f"worst   {scores.min():.4f}")
print(f"best    {scores.max():.4f}")
print(f"spread  {scores.max() - scores.min():.4f}")
print(f"\n5th to 95th percentile: {np.percentile(scores, 5):.4f}"
      f" to {np.percentile(scores, 95):.4f}")
print(f"\nfirst ten seeds: {', '.join(f'{s:.4f}' for s in scores[:10])}")


Between **0.9386 and 1.0000**, from the same model on the same data.

If you had run this once with `random_state=0`, published the number, and
someone else had run it with `random_state=7`, you would have a disagreement
worth six points of accuracy and no way to resolve it.

Look back at lesson 2.6, which measured a random forest against boosting and
found gaps of half a point. That comparison was made on a single split. This
cell explains why the lesson concluded the datasets could not tell the models
apart.


## 2. What cross-validation actually buys

Cross-validation is usually introduced as a way to "use all the data". That is
true and it is not the reason to do it. Measure the reason.

Five-fold, forty times, changing only the shuffling.


In [ ]:
cv_means = []
for seed in range(40):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    cv_means.append(cross_val_score(model(), X, y, cv=cv).mean())
cv_means = np.array(cv_means)

print(f"single split:  mean {scores.mean():.4f}   sd {scores.std():.4f}"
      f"   spread {scores.max() - scores.min():.4f}")
print(f"5-fold:        mean {cv_means.mean():.4f}   sd {cv_means.std():.4f}"
      f"   spread {cv_means.max() - cv_means.min():.4f}")
print(f"\nsame mean to four decimals; the standard deviation is"
      f" {scores.std() / cv_means.std():.1f} times smaller")


Identical means, and the standard deviation falls by a factor of **4.4**.

That is what cross-validation buys: not a better estimate of the average, a
**more stable** one. Every fold's test set is a different fifth, and averaging
five of them cancels most of the luck in which rows you happened to hold back.

So the practical rule, which nothing in this track has said out loud yet:
**report a cross-validated score with its spread, never a single split's
number.** A single split is a lottery ticket, and 0.9386 and 1.0000 were both
in the drum.


## 3. Underfitting and overfitting, in one sweep

`C` in logistic regression is the inverse of regularization strength. Small `C`
means heavy regularization, which means the model is not allowed to fit much.

Sweep it and print both scores, because the shape only makes sense with both.


In [ ]:
CS = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print(f"{'C':>8} {'train':>8} {'CV':>8} {'gap':>8}")
for C in CS:
    train = model(C).fit(X, y).score(X, y)
    held = cross_val_score(model(C), X, y, cv=cv).mean()
    print(f"{C:>8} {train:>8.4f} {held:>8.4f} {train - held:>8.4f}")


Read the two columns as a pair.

**Train accuracy only ever goes up**, 0.9086 to 0.9930. More freedom always fits
the data you can see better. That column can never tell you when to stop.

**Cross-validated accuracy peaks in the middle** and falls off both sides. At
C = 0.001 both are bad, 0.9086 and 0.8910: the model is too constrained to
learn what is there, which is **underfitting**. At C = 1000 train is 0.9930 and
CV is 0.9525, a gap of four points: it has fitted things that are not there,
which is **overfitting**.

Two things not to take away from this. "More regularization is safer" is false,
the best setting is in the middle. And the gap between the columns is a symptom
rather than the thing you are optimising: at C = 0.001 the gap is small and the
model is useless.


## 4. Does the model just need more data?

Overfitting and "not enough data" are the same problem seen from two ends, so
the useful question is whether more rows would help. A learning curve answers
it, and answers it before you go and collect them.


In [ ]:
sizes, train_sc, val_sc = learning_curve(
    model(1000), X, y, cv=cv, train_sizes=np.linspace(0.1, 1.0, 8),
    scoring="accuracy",
)
print(f"the overfitting model, C=1000")
print(f"{'rows':>6} {'train':>8} {'held back':>10} {'gap':>7}")
for n, tr, va in zip(sizes, train_sc.mean(axis=1), val_sc.mean(axis=1)):
    print(f"{int(n):>6} {tr:>8.4f} {va:>10.4f} {tr - va:>7.4f}")


The gap narrows from 0.1196 at 45 rows to 0.0453 at 455, and the held-back
column climbs from 0.8804 to 0.9525. So more data does help this model.

But read the last three rows before ordering any: 338 rows scored 0.9473, 396
scored 0.9508, 455 scored 0.9525. Fifty-nine more rows bought under two
thousandths, and the column is not even monotonic on the way there, dipping to
0.9367 at 279. That is the answer a learning curve gives you: more data helps,
and it has stopped being the cheapest thing to buy.

Which is what the curve is for. Had the held-back column gone flat with the gap
still wide, more rows would be wasted money and the fix would be a simpler
model or better features. Here it is flattening rather than flat, so the
honest reading is that regularization, not collection, is the lever.


## 5. The mistake that survives all of the above

Here is the one to watch for, because it looks like careful practice.

Try seven values of `C`, keep the one that cross-validates best, report that
score. Everything about that sounds right. The problem is that the score was
chosen for being high on those particular folds, so some of its height is luck
that will not repeat.

**Nested cross-validation** measures how much. The inner loop picks `C`; the
outer loop scores the whole picking procedure on folds the inner loop never
saw.


In [ ]:
GRID = {"logisticregression__C": CS}
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

search = GridSearchCV(model(), GRID, cv=inner, scoring="accuracy").fit(X, y)
nested = cross_val_score(
    GridSearchCV(model(), GRID, cv=inner, scoring="accuracy"), X, y, cv=outer
)

print(f"flat CV, best of {len(CS)}:  C={search.best_params_['logisticregression__C']}"
      f"   score {search.best_score_:.4f}")
print(f"nested CV, same search:   score {nested.mean():.4f}  +/- {nested.std():.4f}")
print(f"optimism:                 {search.best_score_ - nested.mean():+.4f}")


Three and a half thousandths on a grid of seven. Small.

Now do it with a grid of twenty-five and watch it grow.


In [ ]:
svm = make_pipeline(StandardScaler(), SVC())
BIG = {
    "svc__C": [0.01, 0.1, 1, 10, 100],
    "svc__gamma": ["scale", 0.001, 0.01, 0.1, 1],
}
search_svm = GridSearchCV(svm, BIG, cv=inner, scoring="accuracy").fit(X, y)
nested_svm = cross_val_score(GridSearchCV(svm, BIG, cv=inner, scoring="accuracy"),
                             X, y, cv=outer)

print(f"flat CV, best of 25:    {search_svm.best_score_:.4f}"
      f"   {search_svm.best_params_}")
print(f"nested CV:              {nested_svm.mean():.4f}  +/- {nested_svm.std():.4f}")
print(f"optimism:               {search_svm.best_score_ - nested_svm.mean():+.4f}")

print(f"\nnow compare the two models the way somebody would:")
print(f"  by flat CV:    logistic regression {search.best_score_:.4f}"
      f"   tuned SVM {search_svm.best_score_:.4f}"
      f"   -> the SVM wins by {search_svm.best_score_ - search.best_score_:+.4f}")
print(f"  by nested CV:  logistic regression {nested.mean():.4f}"
      f"   tuned SVM {nested_svm.mean():.4f}"
      f"   -> {nested_svm.mean() - nested.mean():+.4f}")


The optimism doubled with the grid, +0.0035 to +0.0070, which is the pattern:
**the more settings you try, the more your winning score is a record of how many
chances you gave it.**

And read those last two lines, because that is the whole lesson landing.

By flat cross-validation the tuned SVM beats logistic regression, 0.9824 to
0.9789. By nested cross-validation they are **identical**, 0.9754 and 0.9754.
The SVM's entire advantage was the optimism from searching a grid five times
larger. Comparing models on the score you tuned them on compares how hard you
searched.

Note the nested spreads too, 0.0140 and 0.0244. Both are wider than any of the
differences being argued about, which is section 1 again in a different costume.


## What to take from this

| Claim | What we measured |
|---|---|
| A single train/test score is a fact about the model | It ranged 0.9386 to 1.0000 across 200 splits of the same data |
| Cross-validation exists to use all the data | True but not the point. It cut the standard deviation by 4.4x |
| A wide train-to-held-back gap is the thing to minimise | No. At C=0.001 the gap was tiny and the model scored 0.8910 |
| More regularization is the safe direction | No. The best setting was in the middle and both ends were worse |
| The best cross-validated score is an honest estimate | No. Optimistic by 0.0035 on a grid of 7 and 0.0070 on a grid of 25 |
| The tuned SVM beat logistic regression | Only on the score it was tuned on. Nested CV put both at 0.9754 |

Three habits, and they cost almost nothing:

1. Report cross-validated scores with a spread, never a single split.
2. If you tuned, quote a nested estimate, or hold back a test set you touch
   exactly once.
3. Before believing any difference, ask whether it is bigger than the spread
   from habit 1.


## Try this

1. Change `test_size` from 0.2 to 0.4 in section 1. The held-back set doubles,
   so the spread should shrink. By how much, and what did you pay for it?
2. Run section 5 with a grid of two `C` values instead of seven. Does the
   optimism shrink towards zero the way the pattern predicts?
3. Take lesson 2.6's forest-against-boosting comparison and redo it with nested
   cross-validation and standard deviations. That lesson concluded a few hundred
   rows could not separate the models. Check it.
